# EDA Users

### Imports y lectura de datos

In [136]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from bs4 import BeautifulSoup
np.random.seed(123)

In [137]:
df = pd.read_xml("../../EnglishSet/Users.xml")
usuariosTrain,usuariosTest = train_test_split(df,test_size=0.3)

## Inspección General e Inicial

In [138]:
print("Dimensiones de los datos:",usuariosTrain.shape)
usuariosTrain.head(5)

Dimensiones de los datos: (278890, 12)


,Id,Reputation,CreationDate,DisplayName,LastAccessDate,WebsiteUrl,Location,AboutMe,Views,UpVotes,DownVotes,AccountId
236077,336496,1,2019-02-16T05:04:42.337,Juan Crespo,2019-02-16T05:04:42.337,None,None,None,0,0,0,15344275.0
222770,302644,1,2018-06-10T10:01:41.317,Stardestiny,2018-06-10T10:01:41.317,None,None,None,0,0,0,13747632.0
253867,369033,1,2019-12-06T14:27:18.077,p1ka2So,2023-01-30T09:47:38.690,None,None,<p>Like to play chess and bike rides.</p>\n,0,0,0,6475942.0
328166,417224,1,2021-03-11T20:56:21.120,TeenGoose,2022-10-25T17:52:16.200,NaN,"Buenos Aires, Argentina",NaN,0,0,0,20844213.0
198190,279364,1,2018-02-01T13:59:51.020,Waqar Ali Shah,2019-12-29T19:16:24.383,None,None,None,3,0,0,7615573.0


No aparecen todos los campos, faltan 2, la imagen de perfil y el emailHash. Ninguna de las dos nos iba a ser de utilidad, no pasa nada que no estén. De igual forma, el ID y AccountID en principio no nos sirven de nada ya que son identificadores que no son útiles. También se ve que los campos de fecha están mal y la estructura de los AboutMe no es un string al uso.

In [139]:
usuariosTrain.dtypes

Id                  int64
Reputation          int64
CreationDate       object
DisplayName        object
LastAccessDate     object
WebsiteUrl         object
Location           object
AboutMe            object
Views               int64
UpVotes             int64
DownVotes           int64
AccountId         float64
dtype: object

Hay que cambiar tipo de las fechas y tal vez sea buena idea ver el comportamiento de Location y tal vez separarlo en ciudad y pais. También parece no comportarse siempre de la misma manera en el AboutMe (aparece un NaN y un None). Esto puede ser casualidad y que la descripción de esa persona en concreto sea NaN.

### AboutMe

Este campo tiene una estructura rara como se dijo anteriormente, así que se verán algunos ejemplos para ver exáctamente qué forma tiene

In [140]:
muestraAboutMe = usuariosTrain.query('AboutMe.notna()')["AboutMe"].head().tolist()

for muestra in muestraAboutMe:
    print(muestra)

<p>Like to play chess and bike rides.</p>

<p><a href="https://i.stack.imgur.com/Xyzz4.png" rel="nofollow noreferrer"><img src="https://i.stack.imgur.com/Xyzz4.png" alt="enter image description here" /></a></p>

<p>Currently an indie video game developer. I also love Python 3, Django, REST, and PostgreSQL.</p>



<p><a href="https://www.gofundme.com/f/stop-stack-overflow-from-defaming-its-users" rel="nofollow noreferrer"><img src="https://i.stack.imgur.com/zktfr.png?s=328&amp;g=1" alt="Fund efforts to stop the harm to Monica Cellio" title="Additional funds appreciated"></a></p>

<p><a href="https://stackexchange.com/users/1048581/michaelb958?tab=activity"><img src="https://stackexchange.com/users/flair/1048581.png?theme=dark" alt="profile for michaelb958 on Stack Exchange, a network of free, community-driven Q&amp;A sites" title="Look, I&#39;m on Stack Exchange!"></a></p>



<p>The competition stage.
By the end of the quarter, netflix subscribers of 208 million, is still the boss of th

Este formato es HTML, como solo nos interesa realmente el texto, usaremos una herramienta llamada BeautifulSoup para poder limpiar todo lo que nos importe eliminando las tags, enlaces, imágenes etc.

## Nulls NaNs Nones

In [141]:
usuariosTrain.isna().sum()

Id                     0
Reputation             0
CreationDate           0
DisplayName           21
LastAccessDate         0
WebsiteUrl        241850
Location          207713
AboutMe           219057
Views                  0
UpVotes                0
DownVotes              0
AccountId             14
dtype: int64

Existen unos cuantos nulos en DisplayName, lo que no tiene mucho sentido, esto debe ser o un error de lectura o que tal vez los usuariosTrain hayan desaparecido por x razón. De igual forma hay un comportamiento extraño en los AccountId que tampoco deberían de ser nulos si son identificadores generados automáticamente.

### AccountId NaN

In [142]:
usuariosTrain_NaNAccountId = usuariosTrain[usuariosTrain.isna()["AccountId"]]
usuariosTrain_NaNAccountId

,Id,Reputation,CreationDate,DisplayName,LastAccessDate,WebsiteUrl,Location,AboutMe,Views,UpVotes,DownVotes,AccountId
176270,292687,1,2018-04-10T22:26:32.647,Gaurov bansal,2018-04-10T22:26:32.647,None,None,None,0,0,0,NaN
341824,447148,1,2022-02-20T10:49:33.670,a25bedc5-3d09-41b8-82fb-ea6c353d75ae,2022-02-20T10:49:33.670,None,None,None,1,0,0,NaN
289215,324390,1,2018-11-15T13:26:30.023,Ahsan Sial,2018-11-15T13:26:30.023,None,None,None,0,0,0,NaN
138710,249255,1,2017-07-24T16:52:56.593,Daniel Sasson,2017-07-24T16:52:56.593,None,None,None,1,0,0,NaN
139924,220439,1,2017-02-15T06:58:35.247,a25bedc5-3d09-41b8-82fb-ea6c353d75ae,2017-02-15T06:58:35.247,None,None,None,0,0,0,NaN
228533,355947,1,2019-07-27T23:37:12.837,Abdallah Ghareeb,2019-07-27T23:37:12.837,None,None,None,1,0,0,NaN
262322,321198,1,2018-10-22T14:01:40.140,a25bedc5-3d09-41b8-82fb-ea6c353d75ae,2018-10-22T14:01:40.140,None,None,None,0,0,0,NaN
370661,473820,1,2023-02-17T14:53:21.777,English learner,2023-02-17T14:53:21.777,None,None,None,0,0,0,NaN
314394,444083,1,2022-01-12T17:16:45.620,a25bedc5-3d09-41b8-82fb-ea6c353d75ae,2022-01-12T17:16:45.620,None,None,None,0,0,0,NaN
161634,255416,1,2017-09-04T04:22:16.387,Ali Brauda,2017-09-04T04:22:16.387,None,None,None,1,0,0,NaN


Todos parecen tener un comportamiento muy similar en los casos de que AccountId sea NaN, muchos comparten el mismo DisplayName "a25bedc5-3d09-41b8-82fb-ea6c353d75ae", todos tienen la misma reputación, no tienen ningún campo en WebsiteUrl, Location, AboutMe, ambos campos de votos...

Pero lo que más curioso parece ser es que las fechas de creación y los últimos accesos coinciden aparentemente así a primera vista.

In [143]:
(usuariosTrain_NaNAccountId["CreationDate"] == usuariosTrain_NaNAccountId["LastAccessDate"]).sum()

np.int64(14)

Efectivamente, estas fechas coinciden para los 17 usuariosTrain. Tal vez esto sea porque son cuentas que se empezaron a crear pero nunca terminaron de verificarse o algún tipo de error raro en el sistema. Sin embargo, es raro que algunas de estas cuentas tengan Views.

### DisplayName None

In [144]:
usuariosTrain_DisplayName = usuariosTrain[usuariosTrain.isna()["DisplayName"]]
usuariosTrain_DisplayName.tail(10)

,Id,Reputation,CreationDate,DisplayName,LastAccessDate,WebsiteUrl,Location,AboutMe,Views,UpVotes,DownVotes,AccountId
21132,24693,101,2012-08-09T15:11:07.640,NaN,2021-10-06T13:48:02.590,https://inkrement.ai,"Vienna, Austria",<p>Postdoctoral researcher at the WU Vienna wi...,0,1,0,301763.0
50756,82824,41,2014-07-02T19:17:36.303,NaN,2024-03-28T19:38:57.287,None,"North Carolina, United States",None,12,2,0,297073.0
153132,222381,1,2017-02-26T03:34:59.577,NaN,2021-08-07T07:04:18.900,None,None,None,0,0,0,5128885.0
274919,387607,1,2020-06-04T10:37:29.680,NaN,2020-06-04T16:02:27.640,None,None,None,0,0,0,18760072.0
127472,218718,31,2017-02-05T06:20:49.883,NaN,2017-02-05T07:58:51.893,None,None,None,1,0,0,10181472.0
48272,78376,101,2014-06-05T11:13:31.337,NaN,2023-02-06T00:49:13.957,http://mothteeth.com,New Zealand,"<p>I am a game developer from New Zealand, wor...",10,2,0,122868.0
18343,21658,101,2012-05-28T09:52:25.053,NaN,2017-09-18T23:44:47.167,http://www.stats.stackexchange.com,India,<p>I am a professional with interests in data ...,2,0,0,1521894.0
46420,75348,185,2014-05-16T04:36:22.380,NaN,2023-05-12T05:12:41.443,None,None,None,9,5,0,4468448.0
135023,219909,1,2017-02-12T06:16:55.823,NaN,2017-02-12T21:08:05.633,None,None,None,1,0,0,10232209.0
123233,218129,1,2017-02-02T03:13:00.837,NaN,2017-02-02T03:57:27.663,None,None,None,0,0,0,10159990.0


No parece haber patrones, pero sí que se ve algo que ya se mencionó anteriormente, los NaN como None. Como primer acercamiento vamos a ve al usuario con Id == 97692 para ver si toma como nulos los NaN. Dependiendo de lo que aparezca veremos si también hay que hacer algo con eso o no.

### NaN == Null ???

In [145]:
print(usuariosTrain_DisplayName[usuariosTrain_DisplayName["Id"] == 219909].isna().sum())
usuariosTrain_DisplayName[usuariosTrain_DisplayName["Id"] == 219909]

Id                0
Reputation        0
CreationDate      0
DisplayName       1
LastAccessDate    0
WebsiteUrl        1
Location          1
AboutMe           1
Views             0
UpVotes           0
DownVotes         0
AccountId         0
dtype: int64


,Id,Reputation,CreationDate,DisplayName,LastAccessDate,WebsiteUrl,Location,AboutMe,Views,UpVotes,DownVotes,AccountId
135023,219909,1,2017-02-12T06:16:55.823,NaN,2017-02-12T21:08:05.633,None,None,None,1,0,0,10232209.0


Efectivamente parece tomarlos como nulos. Lo vemos tanto en el campo de WebsiteUrl como en el campo Location. De todas formas, mantendremos un formato similar para todos los datos y los cambiaremos todos a None, que es el formato estándar que suele usar python.

In [146]:
usuariosTrain = usuariosTrain.fillna(pd.NA)
usuariosTrain.query('Id == 219909') #usuariosTrain[usuariosTrain["Id"] == 219909]


,Id,Reputation,CreationDate,DisplayName,LastAccessDate,WebsiteUrl,Location,AboutMe,Views,UpVotes,DownVotes,AccountId
135023,219909,1,2017-02-12T06:16:55.823,<NA>,2017-02-12T21:08:05.633,<NA>,<NA>,<NA>,1,0,0,10232209.0


## Transformación y limpieza inicial

Lo primero y más evidente es usar un formato de tipos de datos acorde a lo que debería ser. También eliminaremos las observaciones que ya vimos que tenían AccountId=Null dado que son casos muy muy escasos, con un patrón muy marcado y que no van a aportar nada al modelo más allá de tener un impacto negativo. También eliminaremos en general el campo AccountId (A diferencia del campo Id, que lo conservaremos en caso de que queramos conectar entre sí las tablas de post y users a futuro, dado que es el campo que se usa como clave foranea para conectar estas dos tablas). 

In [147]:
def AccountID_Remover(dataframe):
    """
    Elimina las filas donde AccountId es nulo y elimina la columna AccountId del dataframe.
    """
    dataframe = dataframe.dropna(subset=['AccountId'])  # Eliminar filas con AccountId nulo
    dataframe = dataframe.drop(columns=['AccountId'], errors='ignore')  # Eliminar la columna AccountId
    return dataframe

def Datetime_Converter(dataframe):
    """
    Convierte las columnas CreationDate y LastAccessDate de string a datetime,
    omitiendo los milisegundos.
    """
    dataframe['CreationDate'] = pd.to_datetime(dataframe['CreationDate']).astype('datetime64[s]')
    dataframe['LastAccessDate'] = pd.to_datetime(dataframe['LastAccessDate']).astype('datetime64[s]')
    return dataframe


In [148]:
usuariosTrain = AccountID_Remover(usuariosTrain)
usuariosTrain = Datetime_Converter(usuariosTrain)
usuariosTrain.head(5)

,Id,Reputation,CreationDate,DisplayName,LastAccessDate,WebsiteUrl,Location,AboutMe,Views,UpVotes,DownVotes
236077,336496,1,2019-02-16 05:04:42,Juan Crespo,2019-02-16 05:04:42,<NA>,<NA>,<NA>,0,0,0
222770,302644,1,2018-06-10 10:01:41,Stardestiny,2018-06-10 10:01:41,<NA>,<NA>,<NA>,0,0,0
253867,369033,1,2019-12-06 14:27:18,p1ka2So,2023-01-30 09:47:38,<NA>,<NA>,<p>Like to play chess and bike rides.</p>\n,0,0,0
328166,417224,1,2021-03-11 20:56:21,TeenGoose,2022-10-25 17:52:16,<NA>,"Buenos Aires, Argentina",<NA>,0,0,0
198190,279364,1,2018-02-01 13:59:51,Waqar Ali Shah,2019-12-29 19:16:24,<NA>,<NA>,<NA>,3,0,0


In [149]:
print("Dimensiones Datos:",usuariosTrain.shape)
usuariosTrain.dtypes

Dimensiones Datos: (278876, 11)


Id                        int64
Reputation                int64
CreationDate      datetime64[s]
DisplayName              object
LastAccessDate    datetime64[s]
WebsiteUrl               object
Location                 object
AboutMe                  object
Views                     int64
UpVotes                   int64
DownVotes                 int64
dtype: object

Como ya mostramos en anteriormente, también hay que transformar las columnas de AboutMe para que sean 100% legibles, y dejar como NA los que sean HTML con 0 texto

In [150]:
def HTML_to_Text(dataframe):   
    dataframe['AboutMe'] = dataframe['AboutMe'].apply(
    lambda muestra: BeautifulSoup(muestra, "html.parser").get_text().strip() if not (muestra is pd.NA) else pd.NA
    )
    dataframe["AboutMe"] = dataframe["AboutMe"].replace("",pd.NA)
    return dataframe

In [151]:
usuariosTrain = HTML_to_Text(usuariosTrain)

/tmp/ipykernel_4595/2558609863.py:3: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  lambda muestra: BeautifulSoup(muestra, "html.parser").get_text().strip() if not (muestra is pd.NA) else pd.NA


In [152]:
usuariosTrain[["AboutMe"]].query("AboutMe.notna()").head()

,AboutMe
253867,Like to play chess and bike rides.
23505,Currently an indie video game developer. I als...
303293,The competition stage.\nBy the end of the quar...
160517,Software Developer - keith.nicholas@gmail.com\...
371396,Self-taught developer and passionate for life ...
